# Double Sorting Robustness: Size-Controlled Anomaly Returns

## Overview
This Jupyter Notebook performs conditional double sorts to evaluate whether fundamental anomalies persist after controlling for firm size. It analyzes 5×5 double sorts, first sorting stocks by market capitalization, then by signal, computing annualized excess returns and Newey-West t-statistics.

## Key Features
- **Automated Signal Selection**: Loads results from single-sort regressions, selects signals with significant FF6 alphas, prioritizing "Cash Cushion".
- **Data Processing**: Converts deciles to quintiles, filters dates, computes matrices of returns and t-stats.
- **Spreads Calculation**: High-Low spreads per size quintile and Small-Big spreads per signal quintile, with HAC-adjusted errors.
- **Results Display**: Console output with significance stars, LaTeX table generation for panels and summary.
- **Summary Table**: Ranks signals by average H-L spread across sizes, exports to LaTeX.

## Methodology
1. Load double-sort Parquet files for each signal.
2. Pivot data into 5×5 matrices for value-weighted returns.
3. Compute Newey-West t-stats (12 lags) for cells and spreads.
4. Display panels for selected signals, generate multi-panel LaTeX tables.
5. Summarize all signals, focusing on size-conditioned H-L spreads.

## Dependencies
- Libraries: pandas, numpy, statsmodels, pathlib.
- Data: Double-sort Parquet files, regression results Parquet.
- Environment: macOS with VS Code, Python 3.x.

## Usage
1. Update CONFIG for paths and parameters.
2. Run cells to load and process data automatically.
3. Review console outputs and generated LaTeX files in results/latex/.
4. Adjust LAYOUT_CONFIG for table formatting.

## Output Files
- **results/latex/appendix_double_sorts_conditional.tex**: Multi-panel LaTeX table with double-sort matrices for selected signals.
- **results/latex/results_3_table_double_sorts_summary.tex**: Summary LaTeX table ranking signals by average H-L spreads across size quintiles.

This notebook provides robust checks for anomaly persistence across firm sizes, bridging data analysis and academic reporting.

In [3]:
# Imports
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
import warnings

# Settings
warnings.filterwarnings('ignore')

## Configuration

In [4]:
CONFIG = {
    'data_dir': Path('data/portfolios/doublesorting'),
    'ff_factors_file': Path('data/famaFactors/FF5.csv'),
    'start_date': '1963-07-01',
    'end_date': '2024-06-30',
    'newey_west_lags': 12,
    'min_obs': 60
}

In [5]:
# Mapping manuel : snake_case → Nom académique propre
SIGNAL_NAME_MAP = {
    # Leverage & Solvency
    'debt_maturity': 'Debt Maturity',
    'leverage_efficiency': 'Leverage Efficiency',
    
    # Profitability
    'op_margin_persist': 'OP Margin Persist',
    'earnings_quality': 'Earnings Quality',
    'ebitda_margin': 'EBITDA Margin',
    
    # Asset Efficiency
    'intangible_power': 'Intangible Power',
    'inv_margin': 'Inventory Margin',
    'receivables_turnover': 'Receivables Turnover',
    'ppe_productivity': 'PPE Productivity',
    'asset_turnover_quality': 'Asset Turnover Quality',
    
    # Liquidity
    'cash_cushion': 'Cash Cushion',
    
    # Capital Intensity
    'low_capex_high_margin': 'Low Capex High Margin',
    'depreciation_efficiency': 'Depreciation Efficiency',
    
    # Quality
    'gp_persist': 'GP Persist',
    'asset_tangibility': 'Asset Tangibility',
    'earnings_smoothness': 'Earnings Smoothness',
    
    # Accruals
    'noa_change': 'NOA Change',
    
    # Growth & Dynamics
    'earnings_accel': 'Earnings Accel',
    'sales_accel': 'Sales Acceleration',
    'recv_vs_sales': 'Recv vs Sales Growth',
    'growth_exhaustion': 'Growth Exhaustion',
    
    # Operating Leverage
    'sga_gp_leverage': 'SGA GP Leverage',
    
    # Returns Quality
    'investment_quality': 'Investment Quality',
    'roic_momentum': 'ROIC Momentum',
    'margin_stability_new': 'Margin Stability'
}

def clean_signal_name(signal_name: str) -> str:
    """
    Convertit nom technique en nom académique propre.
    
    Priority:
        1. Utiliser mapping manuel (SIGNAL_NAME_MAP)
        2. Fallback: snake_case → Title Case automatique
    
    Examples:
        'cash_cushion' → 'Cash Cushion'
        'noa_change' → 'NOA Change'
        'op_margin_persist' → 'OP Margin Persist'
    """
    if signal_name in SIGNAL_NAME_MAP:
        return SIGNAL_NAME_MAP[signal_name]
    else:
        # Fallback: Title Case basique
        return signal_name.replace('_', ' ').title()

## Load and Process Double-Sort Data

Load double-sorted portfolio returns and calculate the matrix of annualized excess returns and t-statistics.

In [6]:
def load_and_process_double_sort(signal: str) -> dict:
    """
    Charge et traite un double sort 5×5 (VERSION NEWEY-WEST).
    
    Returns:
        dict avec matrix_returns, matrix_tstats, spread_hl, spread_hl_tstat, spread_sb, spread_sb_tstat, n_obs
    """
    
    # Charger fichier
    file_path = CONFIG['data_dir'] / f"{signal}_10_90.parquet"
    df = pd.read_parquet(file_path)
    
    # Renommer colonnes
    column_mapping = {
        'mthdate': 'date',
        'size_bin': 'size_quintile',
        'signal_bin': 'signal_decile'
    }
    df = df.rename(columns=column_mapping)
    
    # Conversion déciles → quintiles
    df['signal_quintile'] = ((df['signal_decile'] - 1) // 2) + 1
    
    # Normaliser dates
    df['date'] = pd.to_datetime(df['date']) + pd.offsets.MonthEnd(0)
    
    # Filtrer période
    df = df[(df['date'] >= CONFIG['start_date']) & (df['date'] <= CONFIG['end_date'])]
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : MATRICE 5×5 (Rendements Moyens VW)
    # ─────────────────────────────────────────────────────────────────────
    
    pivot = df.pivot_table(
        index='size_quintile',
        columns='signal_quintile',
        values='ret_vw',
        aggfunc='mean'
    )
    
    matrix_returns = pivot * 12 * 100  # Annualiser
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : T-STATS PAR CELLULE (NEWEY-WEST)
    # ─────────────────────────────────────────────────────────────────────
    
    def compute_tstat(group):
        """Calcule t-stat Newey-West avec 12 lags."""
        ret = group['ret_vw'].dropna()
        if len(ret) < CONFIG['min_obs']:
            return np.nan
        
        X = sm.add_constant(np.ones(len(ret)))
        y = ret.values
        
        try:
            model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': CONFIG['newey_west_lags']})
            tstat = model.tvalues[0]
            return tstat
        except:
            return np.nan

    tstats = df.groupby(['size_quintile', 'signal_quintile']).apply(compute_tstat, include_groups=False)
    matrix_tstats = tstats.unstack()
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : SPREAD HIGH-LOW (par Size Quintile) - NEWEY-WEST
    # ─────────────────────────────────────────────────────────────────────
    
    spread_hl = matrix_returns[5] - matrix_returns[1]
    
    spread_hl_tstat = {}
    for size_q in [1, 2, 3, 4, 5]:
        df_size = df[df['size_quintile'] == size_q]
        
        df_size_pivot = df_size.pivot_table(
            index='date',
            columns='signal_quintile',
            values='ret_vw'
        )
        
        if 1 not in df_size_pivot.columns or 5 not in df_size_pivot.columns:
            spread_hl_tstat[size_q] = np.nan
            continue
        
        spread = df_size_pivot[5] - df_size_pivot[1]
        spread = spread.dropna()
        
        if len(spread) < CONFIG['min_obs']:
            spread_hl_tstat[size_q] = np.nan
        else:
            X = sm.add_constant(np.ones(len(spread)))
            y = spread.values
            try:
                model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': CONFIG['newey_west_lags']})
                spread_hl_tstat[size_q] = model.tvalues[0]
            except:
                spread_hl_tstat[size_q] = np.nan
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 4 : SPREAD SMALL-BIG (par Signal Quintile) - NEWEY-WEST
    # ─────────────────────────────────────────────────────────────────────
    
    spread_sb = matrix_returns.loc[1] - matrix_returns.loc[5]
    
    spread_sb_tstat = {}
    for signal_q in [1, 2, 3, 4, 5]:
        df_signal = df[df['signal_quintile'] == signal_q]
        
        df_signal_pivot = df_signal.pivot_table(
            index='date',
            columns='size_quintile',
            values='ret_vw'
        )
        
        if 1 not in df_signal_pivot.columns or 5 not in df_signal_pivot.columns:
            spread_sb_tstat[signal_q] = np.nan
            continue
        
        spread = df_signal_pivot[1] - df_signal_pivot[5]
        spread = spread.dropna()
        
        if len(spread) < CONFIG['min_obs']:
            spread_sb_tstat[signal_q] = np.nan
        else:
            X = sm.add_constant(np.ones(len(spread)))
            y = spread.values
            try:
                model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': CONFIG['newey_west_lags']})
                spread_sb_tstat[signal_q] = model.tvalues[0]
            except:
                spread_sb_tstat[signal_q] = np.nan
    
    return {
        'matrix_returns': matrix_returns,
        'matrix_tstats': matrix_tstats,
        'spread_hl': spread_hl,
        'spread_hl_tstat': pd.Series(spread_hl_tstat),
        'spread_sb': spread_sb,
        'spread_sb_tstat': pd.Series(spread_sb_tstat),
        'n_obs': len(df['date'].unique())
    }

## Process Target Signals

Process **Cash Cushion** (best-performing anomaly) and **ROIC Momentum** (worst-performing anomaly).

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# 🔍 ÉTAPE 1 : ✅ SÉLECTION AUTOMATIQUE (Alpha FF6 significatif)
# ═══════════════════════════════════════════════════════════════════════

# Charger résultats régressions
results_path = 'results/latex/regressions/results_regression_all_signals.parquet'
df_results = pd.read_parquet(results_path)

# ═══════════════════════════════════════════════════════════════════════
# ✅ CRITÈRE : Alpha FF6 significatif (|t-stat| > 3) + CASH CUSHION
# ═══════════════════════════════════════════════════════════════════════

df_ff6 = df_results[df_results['model'] == 'FF6'].copy()

# Sélectionner signaux avec |t-stat| > 3 sur alpha FF6
significant_signals_ff6 = df_ff6[df_ff6['alpha_tstat'].abs() > 3.0]['signal'].unique().tolist()

# ✅ FORCER CASH CUSHION EN PREMIER (même si t-stat < 3)
if 'cash_cushion' in significant_signals_ff6:
    significant_signals_ff6.remove('cash_cushion')

# Insérer cash_cushion en première position
TARGET_SIGNALS = ['cash_cushion'] + sorted(significant_signals_ff6)

print(f"\n{'='*80}")
print(f"📊 SIGNAUX SÉLECTIONNÉS (Cash Cushion + |t-stat FF6| > 3) : {len(TARGET_SIGNALS)}")
print(f"{'='*80}")

# Afficher avec alpha annualisé
for signal in TARGET_SIGNALS:
    if signal in df_ff6['signal'].values:
        ff6_data = df_ff6[df_ff6['signal'] == signal].iloc[0]
        alpha_annual = ff6_data['alpha'] * 12 * 100
        tstat = ff6_data['alpha_tstat']
        flag = "⭐ FORCED" if signal == 'cash_cushion' and abs(tstat) <= 3.0 else ""
        print(f"   • {signal:<30} | α_FF6 = {alpha_annual:+7.2f}% | t = {tstat:+6.2f} {flag}")

# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════

TSTAT_THRESHOLDS = {
    '***': 2.576,
    '**': 1.96,
    '*': 1.645
}

print(f"\n🎯 Liste finale : {len(TARGET_SIGNALS)} signaux")
print(f"   → Les double sorts vont être chargés automatiquement dans la cellule suivante")


📊 SIGNAUX SÉLECTIONNÉS (Cash Cushion + |t-stat FF6| > 3) : 8
   • cash_cushion                   | α_FF6 =   +5.89% | t =  +2.30 ⭐ FORCED
   • earnings_accel                 | α_FF6 =   -5.67% | t =  -3.03 
   • earnings_smoothness            | α_FF6 =   -8.37% | t =  -4.27 
   • leverage_efficiency            | α_FF6 =   -5.23% | t =  -3.02 
   • margin_stability_new           | α_FF6 =   -9.11% | t =  -4.09 
   • op_margin_persist              | α_FF6 =   -7.67% | t =  -3.39 
   • roic_momentum                  | α_FF6 =   -8.74% | t =  -4.11 
   • sales_accel                    | α_FF6 =   -8.41% | t =  -3.44 

🎯 Liste finale : 8 signaux
   → Les double sorts vont être chargés automatiquement dans la cellule suivante


## Results Display (Target Signals)

Display the double-sorted results for the selected signals.

In [8]:
def add_stars(tstat: float) -> str:
    """Add significance stars based on t-statistic."""
    abs_t = abs(tstat)
    if abs_t >= 2.576: return '***'
    elif abs_t >= 1.96: return '**'
    elif abs_t >= 1.645: return '*'
    return ''

def display_panel(signal_key: str, signal_label: str, panel_letter: str, data_source=None):
    """Display a single panel of double-sort results."""
    if data_source is None:
        data = results[signal_key]
    else:
        data = data_source
        
    print(f"\n{'='*80}")
    print(f"Panel {panel_letter}: {signal_label}")
    print(f"{'='*80}")
    print()
    
    # Header
    print(f"{'':15} {'Signal Quintile':^50}")
    print(f"{'':15} {'Low (1)':>8} {'2':>7} {'3':>7} {'4':>7} {'High (5)':>8} {'H-L':>7}")
    print(f"{'Size':<15} {'':>50}")
    
    size_labels = {1: 'Small (1)', 2: '2', 3: '3', 4: '4', 5: 'Big (5)'}
    
    # Matrix rows
    for size_q in [1, 2, 3, 4, 5]:
        # Returns line
        ret_line = f"{size_labels[size_q]:<15}"
        for signal_q in [1, 2, 3, 4, 5]:
            ret = data['matrix_returns'].loc[size_q, signal_q]
            tstat = data['matrix_tstats'].loc[size_q, signal_q]
            stars = add_stars(tstat) if not pd.isna(tstat) else ''
            ret_line += f"{ret:7.2f}{stars:3} " if not pd.isna(ret) else f"{'---':>8} "
        
        # H-L spread
        spread_ret = data['spread_hl'].get(size_q, np.nan)
        spread_tstat = data['spread_hl_tstat'].get(size_q, np.nan)
        stars = add_stars(spread_tstat) if not pd.isna(spread_tstat) else ''
        ret_line += f"{spread_ret:7.2f}{stars:3}" if not pd.isna(spread_ret) else f"{'---':>8}"
        print(ret_line)
        
        # T-stats line
        tstat_line = f"{'':15}"
        for signal_q in [1, 2, 3, 4, 5]:
            tstat = data['matrix_tstats'].loc[size_q, signal_q]
            tstat_line += f"({tstat:5.2f}) " if not pd.isna(tstat) else f"{'':>8} "
        tstat_line += f"({spread_tstat:5.2f})" if not pd.isna(spread_tstat) else f"{'':>8}"
        print(tstat_line)
        
        if size_q < 5:
            print()
    
    # S-B spreads
    print()
    sb_ret_line = f"{'S-B':<15}"
    for signal_q in [1, 2, 3, 4, 5]:
        ret = data['spread_sb'].get(signal_q, np.nan)
        tstat = data['spread_sb_tstat'].get(signal_q, np.nan)
        stars = add_stars(tstat) if not pd.isna(tstat) else ''
        sb_ret_line += f"{ret:7.2f}{stars:3} " if not pd.isna(ret) else f"{'---':>8} "
    print(sb_ret_line)
    
    sb_tstat_line = f"{'':15}"
    for signal_q in [1, 2, 3, 4, 5]:
        tstat = data['spread_sb_tstat'].get(signal_q, np.nan)
        sb_tstat_line += f"({tstat:5.2f}) " if not pd.isna(tstat) else f"{'':>8} "
    print(sb_tstat_line)
    
# ═══════════════════════════════════════════════════════════════════════
# 📥 CHARGEMENT AUTOMATIQUE DES DOUBLE SORTS
# ═══════════════════════════════════════════════════════════════════════

print(f"\n{'='*80}")
print(f"📥 CHARGEMENT AUTOMATIQUE DES DOUBLE SORTS")
print(f"{'='*80}\n")

results = {}
SIGNALS = {}

for idx, signal_name in enumerate(TARGET_SIGNALS, start=1):
    try:
        print(f"🔄 [{idx}/{len(TARGET_SIGNALS)}] {signal_name}...", end=' ')
        data = load_and_process_double_sort(signal_name)
        results[signal_name] = data
        
        # ✅ UTILISER clean_signal_name() pour mapping automatique
        signal_label = clean_signal_name(signal_name)
        SIGNALS[signal_name] = signal_label
        
        print(f"✅ {data['n_obs']} observations")
        
    except Exception as e:
        print(f"❌ Erreur : {e}")

print(f"\n✅ {len(results)}/{len(TARGET_SIGNALS)} signaux chargés avec succès")



# ═══════════════════════════════════════════════════════════════════════
# 📊 AFFICHAGE DE TOUS LES PANELS
# ═══════════════════════════════════════════════════════════════════════

if results:
    panel_letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'  # Lettres A-Z pour panels
    
    for idx, (signal_key, signal_label) in enumerate(SIGNALS.items()):
        if signal_key in results:
            panel_letter = panel_letters[idx]
            display_panel(signal_key, signal_label, panel_letter)
    
    print()
    print("Note: Returns are annualized value-weighted excess returns in %.")
    print("t-statistics (in parentheses) are Newey-West adjusted with 12 lags.")
    print("***, **, * denote significance at the 3.0, 2.0, and 1.65 levels (two-tailed).")
else:
    print("\nNo results to display. Check data files.")


📥 CHARGEMENT AUTOMATIQUE DES DOUBLE SORTS

🔄 [1/8] cash_cushion... ✅ 732 observations
🔄 [2/8] earnings_accel... ✅ 732 observations
🔄 [3/8] earnings_smoothness... ✅ 732 observations
🔄 [4/8] leverage_efficiency... ✅ 732 observations
🔄 [5/8] margin_stability_new... ✅ 732 observations
🔄 [6/8] op_margin_persist... ✅ 732 observations
🔄 [7/8] roic_momentum... ✅ 732 observations
🔄 [8/8] sales_accel... ✅ 732 observations

✅ 8/8 signaux chargés avec succès

Panel A: Cash Cushion

                                 Signal Quintile                  
                 Low (1)       2       3       4 High (5)     H-L
Size                                                              
Small (1)        16.21***   13.87***   17.90***   17.05***   16.43***    0.22   
               ( 5.44) ( 4.37) ( 5.62) ( 6.33) ( 5.92) ( 0.10)

2                14.40***   14.25***   15.41***   17.53***   12.67***   -1.72   
               ( 5.12) ( 5.15) ( 5.44) ( 5.96) ( 4.51) (-0.94)

3                12.66***   12.45*

In [15]:
# ═══════════════════════════════════════════════════════════════════════
# 🎨 PARAMÈTRES DE MISE EN PAGE (MODIFIABLES)
# ═══════════════════════════════════════════════════════════════════════

LAYOUT_CONFIG = {
    'font_size': r'\footnotesize',        # Options: \tiny, \scriptsize, \footnotesize, \small
    'table_width': 1,                 # Largeur totale (0.90 = 90% de la page)
    'horizontal_spacing': '1cm',       # Espacement entre panels (horizontal)
    'vertical_spacing': '1.5cm',         # Espacement entre lignes de panels
    'panels_per_row': 4                  # Nombre de panels par ligne
}

# ═══════════════════════════════════════════════════════════════════════
# 🚀 GÉNÉRATION DU TABLEAU COMPLET
# ═══════════════════════════════════════════════════════════════════════

if results and len(results) >= 2:
    latex_output = []
    
    latex_output.append(r"\begin{table}[htbp]")
    latex_output.append(r"  \centering")
    latex_output.append(r"  \caption{Conditional Double Sorts: Size-Controlled Anomaly Returns}")
    latex_output.append(r"  \label{tab:double_sort_conditional}")
    latex_output.append(r"  ")
    latex_output.append(f"  {LAYOUT_CONFIG['font_size']}")
    
    signal_keys = list(SIGNALS.keys())
    panel_letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    panels_per_row = LAYOUT_CONFIG['panels_per_row']
    h_space = LAYOUT_CONFIG['horizontal_spacing']
    v_space = LAYOUT_CONFIG['vertical_spacing']
    
    # Grouper par lignes
    for row_idx in range(0, len(signal_keys), panels_per_row):
        panels_in_row = signal_keys[row_idx:row_idx+panels_per_row]
        n_panels = len(panels_in_row)
        
        # ═══════════════════════════════════════════════════════════════
        # CALCUL AUTOMATIQUE DE LA LARGEUR
        # ═══════════════════════════════════════════════════════════════
        if n_panels == panels_per_row:
            width = LAYOUT_CONFIG['table_width']
        else:
            # Réduire proportionnellement pour panels incomplets
            width = LAYOUT_CONFIG['table_width'] * (n_panels / panels_per_row)
        
        latex_output.append(rf"  \resizebox{{{width}\textwidth}}{{!}}{{%")
        
        # ═══════════════════════════════════════════════════════════════
        # CONSTRUCTION DU TABULAR
        # ═══════════════════════════════════════════════════════════════
        # Format: c@{\hspace{0.4cm}}c@{\hspace{0.4cm}}c...
        col_sep = rf"@{{\hspace{{{h_space}}}}}"
        col_format = col_sep.join(['c'] * n_panels)
        latex_output.append(rf"  \begin{{tabular}}{{@{{}}{col_format}@{{}}}}")
        
        # ═══════════════════════════════════════════════════════════════
        # AJOUTER LES PANELS
        # ═══════════════════════════════════════════════════════════════
        for idx, signal_key in enumerate(panels_in_row):
            latex_output.append(r"    \begin{tabular}{@{}lcccccc@{}}")
            latex_output.append(generate_latex_panel(
                signal_key,
                SIGNALS[signal_key],
                panel_letters[row_idx + idx]
            ))
            latex_output.append(r"    \end{tabular}")
            
            if idx < n_panels - 1:
                latex_output.append(r"    &")
        
        latex_output.append(r"  \end{tabular}}")
        
        # Espacement entre lignes
        if row_idx + panels_per_row < len(signal_keys):
            latex_output.append(rf"  \vspace{{{v_space}}}")
    
    # Footer
    latex_output.append(r"  ")
    latex_output.append(r"  \begin{minipage}{\textwidth}")
    latex_output.append(r"    \small")
    latex_output.append(r"    \textit{Note:} Returns are annualized value-weighted excess returns in \%.")
    latex_output.append(r"    t-statistics (in parentheses) are Newey-West adjusted with 12 lags.")
    latex_output.append(r"    $^{***}$, $^{**}$, $^{*}$ denote significance at the 1\%, 5\%, and 10\% levels (two-tailed).")
    latex_output.append(r"  \end{minipage}")
    latex_output.append(r"\end{table}")
    
    # Sauvegarder
    output_path = Path('results/latex/appendix_double_sorts_conditional.tex')
    # output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(latex_output))
    
    print(f"\n✅ Tableau LaTeX généré : {output_path}")
    print(f"   → {len(signal_keys)} signaux | {(len(signal_keys) + panels_per_row - 1) // panels_per_row} lignes de panels")
    print(f"   → Largeur : {LAYOUT_CONFIG['table_width']*100:.0f}% | Police : {LAYOUT_CONFIG['font_size']}")
    print(f"   → Espacement H : {h_space} | V : {v_space}")
else:
    print("\n❌ Pas assez de résultats pour générer le tableau LaTeX.")


✅ Tableau LaTeX généré : results/latex/appendix_double_sorts_conditional.tex
   → 8 signaux | 2 lignes de panels
   → Largeur : 100% | Police : \footnotesize
   → Espacement H : 1cm | V : 1.5cm


## Summary of All Signals

Overview of the performance of all 25 signals, focusing on the High-Low spread within Small and Big caps.

In [9]:
# Find all signal files
all_files = list(CONFIG['data_dir'].glob("*_10_90.parquet"))
all_signals_summary = []

print(f"Processing {len(all_files)} signals for summary...")

for file in all_files:
    signal_name = file.stem.replace('_10_90', '')
    
    try:
        data = load_and_process_double_sort(signal_name)
        
        # Extract key metrics
        # Small Cap H-L (Size 1)
        small_hl = data['spread_hl'].get(1, np.nan)
        small_hl_t = data['spread_hl_tstat'].get(1, np.nan)
        
        # Big Cap H-L (Size 5)
        big_hl = data['spread_hl'].get(5, np.nan)
        big_hl_t = data['spread_hl_tstat'].get(5, np.nan)
        
        # Average H-L across size bins
        # We take the mean of the spreads
        spreads = [data['spread_hl'].get(q, np.nan) for q in [1, 2, 3, 4, 5]]
        avg_hl = np.nanmean(spreads)
        
        all_signals_summary.append({
            'Signal': signal_name,
            'Small Cap H-L': small_hl,
            'Small t-stat': small_hl_t,
            'Big Cap H-L': big_hl,
            'Big t-stat': big_hl_t,
            'Avg H-L': avg_hl
        })
        
    except Exception as e:
        # print(f"Error processing {signal_name}: {e}")
        pass

# Create DataFrame
df_summary = pd.DataFrame(all_signals_summary)

# Sort by absolute t-stat of Avg H-L or just Avg H-L. Let's sort by Avg H-L descending.
df_summary = df_summary.sort_values('Avg H-L', ascending=False).reset_index(drop=True)

# Find all signal files
all_files = list(CONFIG['data_dir'].glob("*_10_90.parquet"))
all_signals_summary = []

print(f"Processing {len(all_files)} signals for summary...")

for file in all_files:
    signal_name = file.stem.replace('_10_90', '')
    
    try:
        data = load_and_process_double_sort(signal_name)
        
        # Extract key metrics
        # Small Cap H-L (Size 1)
        small_hl = data['spread_hl'].get(1, np.nan)
        small_hl_t = data['spread_hl_tstat'].get(1, np.nan)
        
        # Big Cap H-L (Size 5)
        big_hl = data['spread_hl'].get(5, np.nan)
        big_hl_t = data['spread_hl_tstat'].get(5, np.nan)
        
        # Average H-L across size bins
        # We take the mean of the spreads
        spreads = [data['spread_hl'].get(q, np.nan) for q in [1, 2, 3, 4, 5]]
        avg_hl = np.nanmean(spreads)
        
        all_signals_summary.append({
            'Signal': signal_name,
            'Small Cap H-L': small_hl,
            'Small t-stat': small_hl_t,
            'Big Cap H-L': big_hl,
            'Big t-stat': big_hl_t,
            'Avg H-L': avg_hl
        })
        
    except Exception as e:
        # print(f"Error processing {signal_name}: {e}")
        pass

# Create DataFrame
df_summary = pd.DataFrame(all_signals_summary)

# Sort by absolute t-stat of Avg H-L or just Avg H-L. Let's sort by Avg H-L descending.
df_summary = df_summary.sort_values('Avg H-L', ascending=False).reset_index(drop=True)

# ...existing code...

# Display Summary Table
print(f"\n{'='*100}")
print(f"SUMMARY OF DOUBLE SORTS (Size-Controlled)")
print(f"{'='*100}")
print(f"{'Signal':<30} {'Small Cap H-L':<15} {'t-stat':<10} {'Big Cap H-L':<15} {'t-stat':<10} {'Avg H-L':<10}")
print(f"{'-'*100}")

for _, row in df_summary.iterrows():
    # ✅ UTILISER clean_signal_name() au lieu de .replace().title()
    sig = clean_signal_name(row['Signal'])
    
    small_hl_str = f"{row['Small Cap H-L']:.2f}{add_stars(row['Small t-stat'])}"
    small_t_str = f"({row['Small t-stat']:.2f})"
    
    big_hl_str = f"{row['Big Cap H-L']:.2f}{add_stars(row['Big t-stat'])}"
    big_t_str = f"({row['Big t-stat']:.2f})"
    
    avg_hl_str = f"{row['Avg H-L']:.2f}"
    
    print(f"{sig:<30} {small_hl_str:<15} {small_t_str:<10} {big_hl_str:<15} {big_t_str:<10} {avg_hl_str:<10}")
    
print(f"{'-'*100}")
print("Note: H-L is the annualized return spread between High and Low signal quintiles.")

Processing 25 signals for summary...
Processing 25 signals for summary...

SUMMARY OF DOUBLE SORTS (Size-Controlled)
Signal                         Small Cap H-L   t-stat     Big Cap H-L     t-stat     Avg H-L   
----------------------------------------------------------------------------------------------------
Receivables Turnover           1.12            (0.48)     0.98            (0.43)     1.67      
Recv vs Sales Growth           1.88            (1.32)     2.55            (1.50)     1.58      
Leverage Efficiency            3.26*           (1.74)     0.60            (0.34)     1.58      
Debt Maturity                  0.86            (1.08)     0.89            (0.82)     1.31      
Intangible Power               1.77            (0.46)     -1.54           (-0.87)    0.78      
Cash Cushion                   0.22            (0.10)     3.18            (1.44)     0.76      
Asset Tangibility              1.26            (0.58)     -0.82           (-0.41)    0.57      
PPE Productivi

In [18]:
# ═══════════════════════════════════════════════════════════════════════
# 📊 EXPORT LATEX : SUMMARY TABLE (SIGNAUX SÉLECTIONNÉS)
# ═══════════════════════════════════════════════════════════════════════

def format_latex_summary_cell(value: float, tstat: float) -> str:
    """
    Formate une cellule du summary avec étoiles LaTeX.
    
    Returns:
        "1.12$^{***}$" si |t| >= 2.576
        "0.98$^{**}$"  si |t| >= 1.96
        "0.76$^{*}$"   si |t| >= 1.645
        "0.53"         sinon
    """
    if pd.isna(value) or pd.isna(tstat):
        return "---"
    
    abs_t = abs(tstat)
    if abs_t >= 2.576:
        stars = "$^{***}$"
    elif abs_t >= 1.96:
        stars = "$^{**}$"
    elif abs_t >= 1.645:
        stars = "$^{*}$"
    else:
        stars = ""
    
    return f"{value:.2f}{stars}"


# ═══════════════════════════════════════════════════════════════════════
# 🔍 FILTRER SUMMARY POUR SIGNAUX SÉLECTIONNÉS
# ═══════════════════════════════════════════════════════════════════════

# Filtrer df_summary pour ne garder que les signaux sélectionnés
df_summary_selected = df_summary[df_summary['Signal'].isin(TARGET_SIGNALS)].copy()

# Réordonner selon TARGET_SIGNALS (Cash Cushion en premier)
df_summary_selected['sort_order'] = df_summary_selected['Signal'].map({sig: i for i, sig in enumerate(TARGET_SIGNALS)})
df_summary_selected = df_summary_selected.sort_values('sort_order').reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════
# 📄 GÉNÉRATION DU TABLEAU LATEX
# ═══════════════════════════════════════════════════════════════════════

latex_lines = []

latex_lines.append(r"\begin{table}[htbp]")
latex_lines.append(r"  \centering")
latex_lines.append(r"  \caption{Double Sorts: High-Low Spreads by Size Quintile (Newey-West 12 Lags)}")
latex_lines.append(r"  \label{tab:double_sort_summary}")
latex_lines.append(r"  \small")
latex_lines.append(r"  \begin{tabular}{@{}lcccccc@{}}")
latex_lines.append(r"    \toprule")
latex_lines.append(r"    & \multicolumn{2}{c}{\textbf{Small Cap}} & \multicolumn{2}{c}{\textbf{Big Cap}} & \multicolumn{2}{c}{\textbf{Average}} \\")
latex_lines.append(r"    \cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}")
latex_lines.append(r"    \textbf{Signal} & \textbf{H--L} & \textbf{t-stat} & \textbf{H--L} & \textbf{t-stat} & \textbf{H--L} & \textbf{Rank} \\")
latex_lines.append(r"    \midrule")

# Ajouter les lignes de données
for idx, row in df_summary_selected.iterrows():
    signal_clean = clean_signal_name(row['Signal'])
    
    # Small Cap
    small_hl = format_latex_summary_cell(row['Small Cap H-L'], row['Small t-stat'])
    small_t = f"({row['Small t-stat']:.2f})" if not pd.isna(row['Small t-stat']) else "---"
    
    # Big Cap
    big_hl = format_latex_summary_cell(row['Big Cap H-L'], row['Big t-stat'])
    big_t = f"({row['Big t-stat']:.2f})" if not pd.isna(row['Big t-stat']) else "---"
    
    # Average (sans t-stat)
    avg_hl = f"{row['Avg H-L']:.2f}" if not pd.isna(row['Avg H-L']) else "---"
    rank = f"{idx + 1}" if not pd.isna(row['Avg H-L']) else "---"
    
    latex_lines.append(f"    {signal_clean} & {small_hl} & {small_t} & {big_hl} & {big_t} & {avg_hl} & {rank} \\\\")

latex_lines.append(r"    \bottomrule")
latex_lines.append(r"  \end{tabular}")
latex_lines.append(r"  ")
latex_lines.append(r"  \begin{minipage}{\textwidth}")
latex_lines.append(r"    \small")
latex_lines.append(r"    \textit{Note:} This table reports annualized value-weighted High--Low (H--L) return spreads (in \%) between the highest and lowest signal quintiles, conditional on size.")
latex_lines.append(r"    Stocks are first sorted into quintiles by market capitalization (Small Cap = quintile 1, Big Cap = quintile 5), then independently sorted by the signal.")
latex_lines.append(r"    t-statistics (in parentheses) are computed using Newey-West standard errors with 12 lags to account for autocorrelation and heteroskedasticity.")
latex_lines.append(r"    $^{***}$, $^{**}$, $^{*}$ denote significance at the 1\%, 5\%, and 10\% levels (two-tailed).")
latex_lines.append(r"    Average H--L is the mean spread across all five size quintiles. Rank is based on this average.")
latex_lines.append(r"  \end{minipage}")
latex_lines.append(r"\end{table}")

# Sauvegarder
output_path_summary = Path('results/latex/results_3_table_double_sorts_summary.tex')
# output_path_summary.parent.mkdir(parents=True, exist_ok=True)

with open(output_path_summary, 'w', encoding='utf-8') as f:
    f.write("\n".join(latex_lines))

print(f"\n✅ Tableau Summary LaTeX généré : {output_path_summary}")
print(f"   → {len(df_summary_selected)} signaux sélectionnés")
print(f"   → Trié par ordre : Cash Cushion + {sorted(TARGET_SIGNALS[1:])}")

# ═══════════════════════════════════════════════════════════════════════
# 📊 AFFICHAGE PREVIEW (Console)
# ═══════════════════════════════════════════════════════════════════════

print(f"\n{'='*100}")
print(f"SUMMARY OF DOUBLE SORTS (Selected Signals)")
print(f"{'='*100}")
print(f"{'Signal':<30} {'Small H-L':<12} {'t-stat':<10} {'Big H-L':<12} {'t-stat':<10} {'Avg H-L':<10} {'Rank':<5}")
print(f"{'-'*100}")

for idx, row in df_summary_selected.iterrows():
    sig = clean_signal_name(row['Signal'])
    
    small_hl_str = f"{row['Small Cap H-L']:.2f}{add_stars(row['Small t-stat'])}"
    small_t_str = f"({row['Small t-stat']:.2f})"
    
    big_hl_str = f"{row['Big Cap H-L']:.2f}{add_stars(row['Big t-stat'])}"
    big_t_str = f"({row['Big t-stat']:.2f})"
    
    avg_hl_str = f"{row['Avg H-L']:.2f}"
    rank_str = f"{idx + 1}"
    
    print(f"{sig:<30} {small_hl_str:<12} {small_t_str:<10} {big_hl_str:<12} {big_t_str:<10} {avg_hl_str:<10} {rank_str:<5}")

print(f"{'-'*100}")
print("Note: H-L is the annualized return spread between High and Low signal quintiles.")
print("Signals ranked by average H-L across all size quintiles.")


✅ Tableau Summary LaTeX généré : results/latex/results_3_table_double_sorts_summary.tex
   → 8 signaux sélectionnés
   → Trié par ordre : Cash Cushion + ['earnings_accel', 'earnings_smoothness', 'leverage_efficiency', 'margin_stability_new', 'op_margin_persist', 'roic_momentum', 'sales_accel']

SUMMARY OF DOUBLE SORTS (Selected Signals)
Signal                         Small H-L    t-stat     Big H-L      t-stat     Avg H-L    Rank 
----------------------------------------------------------------------------------------------------
Cash Cushion                   0.22         (0.10)     3.18         (1.44)     0.76       1    
Earnings Accel                 1.09         (0.68)     -1.76        (-0.79)    -0.06      2    
Earnings Smoothness            -3.38        (-1.35)    0.56         (0.17)     -1.08      3    
Leverage Efficiency            3.26*        (1.74)     0.60         (0.34)     1.58       4    
Margin Stability               0.16         (0.10)     -1.38        (-0.60)    